# Week 9 Lab 1 — Real micro-step DSMC data, zonal learning, and validation

<!-- MIE690A real-step-data validation v5 -->

<!-- FLOWMLLAB_COLAB_LAUNCH_V1 -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ehsan-Roohi/FlowMLLab/blob/main/notebooks/week09/W9_Lab1_Microstep_Zonal_DeepONet_Student.ipynb)

**Runtime:** CPU, normally 1–3 minutes with the included compact data.
**Prerequisites:** case-wise splits, relative error, and neural operators.

This lab turns the Roohi--Mahdavi article *Analysis of the rarefied flow at
micro-step using a DeepONet surrogate model with a physics-guided zonal loss
function* (Microfluidics and Nanofluidics 30:44, published 11 May 2026) into a
controlled classroom experiment.

### Learning outcomes

By the end, you should be able to:

1. distinguish a parameter-to-field operator from pointwise regression;
2. keep every geometry case entirely inside one split;
3. explain why a global mean loss can hide recirculation failure;
4. select a zonal-loss weight using validation cases only;
5. compare article contours with an independently trained classroom model; and
6. distinguish data integrity from independent DSMC verification.


## Data and validation contract — read before code

The nine real DSMC height fields originate in the authors'
[`roohi-step-dnn-mahdavi`](https://github.com/Ehsan-Roohi/roohi-step-dnn-mahdavi)
repository at commit `c3f211376b42b8dc30daad380eaef5e0ab800b5c`. With the
corresponding author's explicit permission, FlowMLLab includes two compact
derivatives whose hashes trace back to all nine source files. The seven
development/validation cases and two held-out tests live in different files.

Three evidence levels stay separate:

- **source data:** the nine published, smoothed DSMC fields;
- **retained article evidence:** the paper's MSE/GMSE/zonal table;
- **notebook result:** a new independent coordinate-network baseline trained
  here. It is not the paper's C-DeepONet checkpoint.

The stored C-DeepONet outputs in the pinned article repository were generated
with a local input patch constructed from the same DSMC target field being
reconstructed. They are therefore data-assisted reconstruction evidence, not
an independently deployable held-out prediction. A nearest-sample baseline on
that target-derived patch reaches 4.40% at H44 and 6.79% at H67, below the
stored model's 4.92% and 8.23%. The independent classroom model below never
receives that patch.

The notebook model uses only $h/H$, $(x,y)$, and known geometry. Development,
validation, and held-out geometry cases remain in separate case-wise groups.


In [ ]:
# FLOWMLLAB_COLAB_BOOTSTRAP_V1
from pathlib import Path as _FlowMLLabPath
import os as _flowmllab_os
import subprocess as _flowmllab_subprocess
import sys as _flowmllab_sys

if "google.colab" in _flowmllab_sys.modules or _flowmllab_os.environ.get("COLAB_RELEASE_TAG"):
    _flowmllab_root = _FlowMLLabPath("/content/FlowMLLab")
    if not (_flowmllab_root / ".git").is_dir():
        _flowmllab_subprocess.run(
            ["git", "clone", "--depth", "1",
             "https://github.com/Ehsan-Roohi/FlowMLLab.git", str(_flowmllab_root)],
            check=True,
        )
    _flowmllab_subprocess.run(
        [_flowmllab_sys.executable, "-m", "pip", "install", "-q", "-e", str(_flowmllab_root)],
        check=True,
    )
    _flowmllab_os.chdir(_flowmllab_root / "notebooks/week09")

from pathlib import Path
import json
import platform
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ModuleNotFoundError:
    display = print

REPO_ROOT = next(
    candidate for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "results/mahdavi_deeponet").is_dir()
)
if str(REPO_ROOT) not in _flowmllab_sys.path:
    _flowmllab_sys.path.insert(0, str(REPO_ROOT))
RESULTS = REPO_ROOT / "results/mahdavi_deeponet"
plt.rcParams.update({"font.size": 11, "axes.labelsize": 12, "legend.fontsize": 9})
print("Python:", platform.python_version())
print("FlowMLLab root:", REPO_ROOT)


In [ ]:
from flowmllab.mahdavi_deeponet import (
    STEP_HEIGHT_DEVELOPMENT_PERCENT,
    STEP_HEIGHT_HELD_OUT_PERCENT,
    STEP_HEIGHT_VALIDATION_PERCENT,
    STEP_SOURCE_COMMIT,
    evaluate_step_coordinate_surrogate,
    fit_step_coordinate_surrogate,
    load_step_height_archive,
    predict_step_coordinate_surrogate,
)

paper = pd.read_csv(RESULTS / "step_paper_evidence.csv")
display(paper.pivot(index="objective", columns="scope", values="reported_error_percent"))

manifest = json.loads((RESULTS / "step_source_manifest.json").read_text())
learning_heights = np.concatenate([
    STEP_HEIGHT_DEVELOPMENT_PERCENT,
    STEP_HEIGHT_VALIDATION_PERCENT,
])
learning_cases = load_step_height_archive(REPO_ROOT, split="learning")
first_case = learning_cases[int(STEP_HEIGHT_DEVELOPMENT_PERCENT[0])]
bounds_m = (
    float(first_case["x"].min()), float(first_case["x"].max()),
    float(first_case["y"].min()), float(first_case["y"].max()),
)
print("source commit:", STEP_SOURCE_COMMIT)
print("opened archive: step_height_learning_7cases.npz only")
print("development:", STEP_HEIGHT_DEVELOPMENT_PERCENT.tolist())
print("validation:", STEP_HEIGHT_VALIDATION_PERCENT.tolist())
print("sealed test:", STEP_HEIGHT_HELD_OUT_PERCENT.tolist())


## 1. Start with the real flow and its geometry

The parent grid has 200 streamwise by 120 transverse locations. Points inside
the solid step are absent, so the number of fluid points changes with $h/H$.
The plot below uses equal physical axis scaling: the step is not stretched to
make the recirculation region look larger.


In [ ]:
def on_parent_grid(case, values):
    xs, ys = np.unique(case["x"]), np.unique(case["y"])
    field = np.full((len(ys), len(xs)), np.nan)
    ix = np.searchsorted(xs, case["x"])
    iy = np.searchsorted(ys, case["y"])
    field[iy, ix] = values
    return xs, ys, field

example = learning_cases[50]
xs, ys, u_grid = on_parent_grid(example, example["u"])
fig, ax = plt.subplots(figsize=(11.0, 3.1), constrained_layout=True)
image = ax.pcolormesh(xs * 1e9, ys * 1e9, np.ma.masked_invalid(u_grid),
                      shading="nearest", cmap="coolwarm")
ax.contour(xs * 1e9, ys * 1e9, np.ma.masked_invalid(u_grid),
           levels=[0.0], colors="black", linewidths=1.0)
ax.set(xlabel="x (nm)", ylabel="y (nm)", title=r"Real smoothed DSMC field, $h/H=0.50$")
ax.set_aspect("equal", adjustable="box")
fig.colorbar(image, ax=ax, label="U (source units)")
plt.show()


## 2. What is—and is not—validated about the DSMC labels

SHA-256, finite values, row counts, and grid topology establish **data
integrity**. They do not independently establish DSMC convergence. The public
repository does not yet document the following items at a level that this
course can independently reproduce:

1. cell size relative to local mean free path;
2. time step relative to local collision time;
3. particles per cell;
4. sampling duration, independent seeds, and confidence intervals;
5. statistical uncertainty near separation/reattachment; and
6. wall reflection model and accommodation coefficients.

Students must report these as provenance gaps—not silently invent values. Also,
the two studies vary Kn at fixed geometry and $h/H$ at fixed Kn=0.01. Joint
$(Kn,h/H)$ generalization has not been demonstrated.


In [ ]:
provenance_gaps = pd.Series(manifest["dsmc_provenance_status"], name="status")
display(provenance_gaps.to_frame())
print("scope:", manifest["study_scope"])


## 3. Define an independent coordinate surrogate

For a geometry parameter $h/H$ and query coordinate $\mathbf y=(x,y)$, a
standard DeepONet has the form

$$
\widehat{G}(h/H)(\mathbf{y})=
\sum_{k=1}^{r} b_k(h/H)\,t_k(\mathbf{y})+b_0.
$$

Here we use a small coordinate MLP as a transparent CPU baseline. Its features
contain only the height ratio, normalized coordinates, and a wall-relative
coordinate computed from the known step geometry. The required assignment
later replaces this baseline with a strict branch/trunk dot product.

The paper defines the recirculation zone from the reference streamwise
velocity, $U<0$, and balances two separately normalized regional errors:

$$
\mathcal L_{\rm zonal}=\alpha\mathcal L_{U<0}+
(1-\alpha)\mathcal L_{U\ge 0}.
$$

For the CPU baseline, exact zonal weighting is implemented by deterministic
stratified resampling: a fraction $\alpha$ of optimizer samples comes from
$U<0$ and $1-\alpha$ from the main-flow zone. Regional weights are estimated
only from development cases.


In [ ]:
def fit_coordinate_surrogate(selected_heights, alpha, *, seed=690, sample_size=60_000):
    return fit_step_coordinate_surrogate(
        learning_cases, selected_heights, alpha, bounds_m=bounds_m,
        seed=seed, sample_size=sample_size,
    )


def predict_case(fitted, height, case):
    return predict_step_coordinate_surrogate(fitted, int(height), case)


def evaluate_model(fitted, selected_heights, case_store):
    return pd.DataFrame(evaluate_step_coordinate_surrogate(
        fitted, case_store, selected_heights,
    ))


## 4. Freeze the case-wise split and selection rule

The source dataset contains exactly nine heights. We reserve H44 and H67 for
the final test, use H33 and H58 for validation, and fit candidate settings only
on H16, H21, H25, H50, and H75. Every point from a geometry remains in one
split. This is stricter than randomly splitting points from the same cases.

**Predeclared rule:** among $\alpha\in\{0.5,0.6,0.7,0.8\}$, choose the
largest vortex improvement whose validation global relative error is no more
than two percentage points worse than the unweighted fit. This prevents a
zonal win purchased by an unlimited global failure.


In [ ]:
baseline_fit = fit_coordinate_surrogate(STEP_HEIGHT_DEVELOPMENT_PERCENT, None)
baseline_validation = evaluate_model(
    baseline_fit, STEP_HEIGHT_VALIDATION_PERCENT, learning_cases,
)
baseline_global = 100 * baseline_validation["full_relative_l2"].mean()
selection_rows = []
candidate_models = {}
for alpha in (.5, .6, .7, .8):
    candidate_models[alpha] = fit_coordinate_surrogate(
        STEP_HEIGHT_DEVELOPMENT_PERCENT, alpha,
    )
    metrics = evaluate_model(
        candidate_models[alpha], STEP_HEIGHT_VALIDATION_PERCENT, learning_cases,
    )
    selection_rows.append({
        "alpha": alpha,
        "validation_global_percent": 100 * metrics["full_relative_l2"].mean(),
        "validation_vortex_percent": 100 * metrics["vortex_relative_l2"].mean(),
    })
selection = pd.DataFrame(selection_rows)
eligible = selection[
    selection["validation_global_percent"] <= baseline_global + 2.0
]
selected_alpha = float(eligible.sort_values("validation_vortex_percent").iloc[0]["alpha"])
display(selection)
print(f"unweighted validation global error: {baseline_global:.3f}%")
print("selected alpha:", selected_alpha)
if selected_alpha != 0.6:
    print("Version-sensitive teaching result: the article used alpha=0.6;",
          "this run selected", selected_alpha)


## Stop: held-out geometry gate

At this point the split, architecture, optimizer, sample budget, seed, candidate
weights, and selection rule are frozen. Write your expected global and vortex
errors for H44 and H67. Only then run the next cell. If any choice changes after
viewing these fields, they are no longer held out.


In [ ]:
final_unweighted = fit_coordinate_surrogate(learning_heights, None)
final_zonal = fit_coordinate_surrogate(learning_heights, selected_alpha)
held_out_cases = load_step_height_archive(REPO_ROOT, split="test")
print("opened archive after freeze: step_height_test_2cases.npz")
unweighted_test = evaluate_model(
    final_unweighted, STEP_HEIGHT_HELD_OUT_PERCENT, held_out_cases,
)
zonal_test = evaluate_model(
    final_zonal, STEP_HEIGHT_HELD_OUT_PERCENT, held_out_cases,
)
comparison = pd.concat([
    unweighted_test.assign(model="unweighted"),
    zonal_test.assign(model=f"zonal alpha={selected_alpha:.1f}"),
], ignore_index=True)
display(comparison[["model", "height_percent", "full_relative_l2", "vortex_relative_l2"]])

fig, axes = plt.subplots(2, 3, figsize=(13.2, 4.7), constrained_layout=True)
for row, height in enumerate(STEP_HEIGHT_HELD_OUT_PERCENT):
    case = held_out_cases[int(height)]
    prediction = predict_case(final_zonal, height, case)
    panels = [case["u"], prediction[:, 0], np.abs(prediction[:, 0] - case["u"])]
    titles = ["DSMC target U", "independent prediction U", "absolute U error"]
    for axis, values, title in zip(axes[row], panels, titles):
        gx, gy, field = on_parent_grid(case, values)
        artist = axis.pcolormesh(gx * 1e9, gy * 1e9, np.ma.masked_invalid(field),
                                 shading="nearest", cmap="coolwarm" if "error" not in title else "magma")
        axis.set_title(f"H{int(height)} — {title}")
        axis.set(xlabel="x (nm)", ylabel="y (nm)")
        axis.set_aspect("equal", adjustable="box")
        fig.colorbar(artist, ax=axis, shrink=.78)
plt.show()


## 5. Reproduce the article cases and compare validation levels

Under the final published paper's numbering, the pinned source checkout retains
DSMC and stored NN fields for Figure 6 at Kn=0.004 and Kn=0.02 and for Figure
15 at H44 and H67. Each reconstruction uses the same coordinates, equal
$x/H$--$y/H$ scaling, a masked solid step, and one color range shared by DSMC
and NN for each velocity component. Error panels are clipped only for display
at their 99th percentile; the CSV retains the maximum error.

Figure 6 also contains Kn=1. The exact DSMC source field is included below, but
the neural panel is not reproducible from the pinned repository because its
stored prediction is absent. The coverage table records that gap explicitly;
no values are inferred from the published raster image.

The first table and images are **retained article evidence**. The second table
and images are the independent H44/H67 result from the frozen classroom model.
Compare topology, the $U=0$ recirculation boundary, reverse-flow IoU, and
reattachment length—not merely color similarity.

In the retained panels, “stored NN” means the article-repository output that
uses the target-derived input patch described above. It must not be read as a
fresh prediction from geometry and Knudsen number alone.

Two source inconsistencies remain explicit:

- the article problem statement lists Kn=0.2, while Figure 6, its discussion,
  the source code, and the stored output use Kn=0.02;
- Figure 6 shows Kn=1, but the pinned repository contains no stored Kn=1
  prediction, so this lesson rebuilds only its DSMC contour and does not
  fabricate the neural comparison.


In [ ]:
article_contours = pd.read_csv(RESULTS / "step_article_contour_metrics.csv")
article_coverage = pd.read_csv(RESULTS / "step_article_case_coverage.csv")
independent_contours = pd.read_csv(RESULTS / "step_independent_contour_metrics.csv")
columns = [
    "case_id", "combined_relative_l2_percent", "vortex_relative_l2_percent",
    "negative_u_iou_percent", "dsmc_reattachment_length_over_L",
]
display(article_contours[columns + ["stored_nn_reattachment_length_over_L"]])
display(article_coverage)
display(independent_contours[columns + ["independent_reattachment_length_over_L"]])

try:
    from IPython.display import Image as _ContourImage
except ModuleNotFoundError:
    _ContourImage = None
if _ContourImage is not None:
    for filename in (
        "step_article_contours/article_figure_06_Kn0p004.png",
        "step_article_contours/article_figure_06_Kn0p02.png",
        "step_article_contours/article_figure_06_Kn1_DSMC_only.png",
        "step_article_contours/article_figure_15_H44.png",
        "step_article_contours/article_figure_15_H67.png",
        "step_independent_contours/held_out_H44_independent.png",
        "step_independent_contours/held_out_H67_independent.png",
    ):
        display(_ContourImage(filename=str(RESULTS / filename)))


## 6. Interpret without mixing evidence levels

The real-data classroom experiment shows a real tradeoff: the selected zonal
objective reduces reverse-flow error, but held-out whole-field error rises from
7.228% to 9.594% at H44 and from 5.934% to 11.267% at H67. The second increase
is nearly a factor of two, not a modest change. The article table reports a
different, retained comparison:
zonal loss changes the reported recirculation-zone error from
14.6135% (MSE) to 11.9413%, while the full-domain value changes from 2.1739%
to 2.2254%.

Those four percentages come from the article table. They are not the notebook
MLP errors and are not the upstream SWAG stored-prediction errors.

### DeepONet implementation exercise

Replace the coordinate MLP with two small Keras networks:

- branch input: one scalar, $h/H$;
- trunk input: two scalars, $(x/H,y/H)$;
- output: dot product of equal-width branch and trunk vectors, with separate
  heads for $U$ and $V$.

Keep complete geometry cases together. Implement regional means before mixing
them with fixed $\alpha$; adaptive $\alpha$ is a separate experiment, not the
fixed-loss paper protocol.

### Required submission

1. the source commit and nine verified hashes;
2. a signed, case-wise split table;
3. the validation-only $\alpha$ sweep;
4. global and reverse-flow metrics for every held-out geometry;
5. one equal-aspect contour locating the largest local error;
6. a branch/trunk diagram and complete case-wise split table; and
7. a DSMC verification checklist that labels unavailable setup values as
   unavailable.
